In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.read.table("uber.bronze.rides_raw")
display(df)

In [0]:
df = spark.sql("select * from uber.bronze.bulk_rides")
rides_schema = df.schema

In [0]:
rides_schema = StructType([StructField('ride_id', StringType(), True), StructField('confirmation_number', StringType(), True), StructField('passenger_id', StringType(), True), StructField('driver_id', StringType(), True), StructField('vehicle_id', StringType(), True), StructField('pickup_location_id', StringType(), True), StructField('dropoff_location_id', StringType(), True), StructField('vehicle_type_id', LongType(), True), StructField('vehicle_make_id', LongType(), True), StructField('payment_method_id', LongType(), True), StructField('ride_status_id', LongType(), True), StructField('pickup_city_id', LongType(), True), StructField('dropoff_city_id', LongType(), True), StructField('cancellation_reason_id', LongType(), True), StructField('passenger_name', StringType(), True), StructField('passenger_email', StringType(), True), StructField('passenger_phone', StringType(), True), StructField('driver_name', StringType(), True), StructField('driver_rating', DoubleType(), True), StructField('driver_phone', StringType(), True), StructField('driver_license', StringType(), True), StructField('vehicle_model', StringType(), True), StructField('vehicle_color', StringType(), True), StructField('license_plate', StringType(), True), StructField('pickup_address', StringType(), True), StructField('pickup_latitude', DoubleType(), True), StructField('pickup_longitude', DoubleType(), True), StructField('dropoff_address', StringType(), True), StructField('dropoff_latitude', DoubleType(), True), StructField('dropoff_longitude', DoubleType(), True), StructField('distance_miles', DoubleType(), True), StructField('duration_minutes', LongType(), True), StructField('booking_timestamp', TimestampType(), True), StructField('pickup_timestamp', StringType(), True), StructField('dropoff_timestamp', StringType(), True), StructField('base_fare', DoubleType(), True), StructField('distance_fare', DoubleType(), True), StructField('time_fare', DoubleType(), True), StructField('surge_multiplier', DoubleType(), True), StructField('subtotal', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('total_fare', DoubleType(), True), StructField('rating', DoubleType(), True)])

In [0]:
df = spark.read.table("uber.bronze.rides_raw")
df_parsed = df.withColumn("parsed_rows",from_json(col("rides"),rides_schema))\
                    .select("parsed_rows.*")
display(df_parsed)

In [0]:
%sql
select * from uber.bronze.stg_rides

In [0]:
%sql
select 
  stg_rides.* 
from
  uber.bronze.stg_rides stg_rides
left join
  uber.bronze.map_vehicle_types map_vehicle_types
on
  stg_rides.vehicle_type_id = map_vehicle_types.vehicle_type_id
left join
  uber.bronze.map_vehicle_makes map_vehicle_makes
on
  stg_rides.vehicle_make_id = map_vehicle_makes.vehicle_make_id


In [0]:
jinja_config = [
    {
        "table" : "uber.bronze.stg_rides stg_rides",
        "select" : "stg_rides.*",
        "where" : ""
    },
    {
        "table" : "uber.bronze.map_vehicle_types map_vehicle_types",
        "select" : "map_vehicle_types.vehicle_type_id,map_vehicle_types.description,map_vehicle_types.per_mile,map_vehicle_types.per_minute",
        "where" : "",
        "on" : "stg_rides.vehicle_type_id = map_vehicle_types.vehicle_type_id"
    },
    {
        "table" : "uber.bronze.map_vehicle_makes map_vehicle_makes",
        "select" : "map_vehicle_makes.vehicle_make_id",
        "where" : "",
        "on" : "stg_rides.vehicle_make_id = map_vehicle_makes.vehicle_make_id"
    },
    {
        "table" : "uber.bronze.map_ride_statuses map_ride_statuses",
        "select" : "map_ride_statuses.ride_status",
        "where" : "",
        "on" : "stg_rides.ride_status_id = map_ride_statuses.ride_status_id"
    },
    {
        "table" : "uber.bronze.map_payment_methods map_payment_methods",
        "select" : "map_payment_methods.payment_method, map_payment_methods.is_card, map_payment_methods.requires_auth",
        "where" : "",
        "on" : "stg_rides.payment_method_id = map_payment_methods.payment_method_id"
    },
    {
        "table" : "uber.bronze.map_cities map_cities",
        "select" : "map_cities.city as pickup_city, map_cities.state, map_cities.region,map_cities.updated_at as city_updated_at",
        "where" : "",
        "on" : "stg_rides.pickup_city_id = map_cities.city_id"
    },
    {
        "table" : "uber.bronze.map_cancellation_reasons map_cancellation_reasons",
        "select" : "map_cancellation_reasons.cancellation_reason",
        "where" : "",
        "on" : "stg_rides.cancellation_reason_id = map_cancellation_reasons.cancellation_reason_id"
    }
]

In [0]:
from jinja2 import Template
jinja_str = """
    SELECT 
        {% for config in jinja_config %}
            {{ config.select}}
            {% if not loop.last %}
                ,
            {% endif %}
        {% endfor %}
    FROM
        {% for config in jinja_config %}
            {% if loop.first %}
                {{ config.table}}
            {% else %}
                LEFT JOIN {{ config.table}} ON {{ config.on}}
            {% endif %}
        {% endfor %}

"""

template = Template(jinja_str)
rendered_template = template.render(jinja_config=jinja_config)
print(rendered_template)

In [0]:
display(spark.sql(rendered_template))

In [0]:
%sql
select current_timestamp()

In [0]:
%sql
select * from uber.bronze.stg_rides

In [0]:
%sql
select * from uber.bronze.silver_obt

In [0]:
%sql
select base_fare,city_updated_at from uber.bronze.silver_obt

In [0]:
%sql
select * from uber.bronze.dim_location

In [0]:
%sql
select fact.ride_id,fact.passenger_id,dim.region from uber.bronze.fact as fact
left join uber.bronze.dim_location as dim
on fact.pickup_city_id = dim.pickup_city_id
and dim.`__END_AT` is null